In [0]:
from datetime import datetime
import time
import numpy as np
import pandas as pd

import pyspark.sql.functions as f
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, DoubleType,
    BooleanType, DateType, ArrayType, MapType
)

In [0]:
import urllib.request

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/afonso-pereira/spark-data/refs/heads/main/data/employees.csv',
    '/Volumes/workspace/default/class/employees.csv'
)

In [0]:
df_employees = spark.read.csv(
    '/Volumes/workspace/default/class/employees.csv',
    header=True,
    inferSchema=True
)
df_employees.display()

In [0]:
# 1. Window – partitionBy + avg
window = Window.partitionBy('department')

df_employees.withColumn(
    'avg_dept_salary',
    f.avg('salary').over(window)
).display()

In [0]:
# 2. row_number vs rank vs dense_rank
window = Window.orderBy(f.desc('salary'))

df_employees.select(
    'name', 'salary',
    f.row_number().over(window).alias('row_number'),
    f.rank().over(window).alias('rank'),
    f.dense_rank().over(window).alias('dense_rank')
).display()

In [0]:
# 3. lead & lag
window = Window.partitionBy('department').orderBy('salary')

(
    df_employees
    .withColumn('next_salary', f.lead(f.col('salary'), offset=1).over(window))
    .withColumn('salary_diff', f.col('next_salary') - f.col('salary'))
    .select('name', 'department', 'salary', 'next_salary', 'salary_diff')
).display()

In [0]:
# Array Type – create sample data
data = [
    ('Sara',     ['Portugal', 'Spain', 'France'],                          1, 2),
    ('John',     ['UK', 'Belgium', 'Netherlands', 'Denmark'],              2, 1),
    ('Steffano', ['Italy', 'Croatia', 'Switzerland', 'Portugal', 'UK'],    2, 5),
    ('Marina',   ['Spain', 'Portugal', 'France', 'UK'],                    4, 1),
]

schema = StructType([
    StructField('name',              StringType(),                True),
    StructField('visited_countries', ArrayType(StringType()),     True),
    StructField('best_idx',          IntegerType(),               True),
    StructField('worst_idx',         IntegerType(),               True),
])

df = spark.createDataFrame(data, schema)
df.printSchema()
df.display()

In [0]:
# element_at & slice
df_best_worst = (
    df
    .withColumn('best_country',  f.element_at(f.col('visited_countries'), f.col('best_idx')))
    .withColumn('worst_country', f.element_at(f.col('visited_countries'), f.col('worst_idx')))
    .withColumn('2nd_3rd',       f.slice(f.col('visited_countries'), start=2, length=2))
)
df_best_worst.display()

In [0]:
# size, array_distinct, array_contains
df_best_worst.withColumn('nr_countries',    f.size(f.col('visited_countries')))              .withColumn('unique_countries', f.array_distinct(f.col('visited_countries')))              .withColumn('been_to_portugal', f.array_contains(f.col('visited_countries'), 'Portugal'))              .display()

In [0]:
# Filter using array_contains
df.filter(f.array_contains(f.col('visited_countries'), 'Portugal')).display()

In [0]:
# Struct Type
data = [
    (1, ('Sara',     30, 'Female')),
    (2, ('John',     35, 'Male')),
    (3, ('Steffano', 40, 'Male')),
    (4, ('Marina',   20, 'Female')),
]

schema = StructType([
    StructField('person_id',   IntegerType(), False),
    StructField('person_info', StructType([
        StructField('name',   StringType(),  True),
        StructField('age',    IntegerType(), True),
        StructField('gender', StringType(),  True),
    ]), True),
])

df_struct = spark.createDataFrame(data, schema)
df_struct.printSchema()
df_struct.display()

In [0]:
# Access struct fields — two equivalent syntaxes
# getField()
df_struct.select(
    'person_id',
    f.col('person_info').getField('name').alias('name'),
    f.col('person_info').getField('age').alias('age'),
).display()

# dot notation (preferred)
df_struct.select('person_id', 'person_info.name', 'person_info.age').display()

In [0]:
# Rebuild struct from columns
df_flat = df_struct.select('person_id', 'person_info.name', 'person_info.age', 'person_info.gender')

df_flat.select(
    f.col('person_id'),
    f.struct('name', 'age', 'gender').alias('person_info')
).display()

In [0]:
# Map Type
data = [
    ('Sara',     {'Maths': 90, 'Science': 75, 'History': 95}),
    ('John',     {'Biology': 85, 'Maths': 75, 'Chemistry': 88, 'Literature': 75}),
    ('Stefanno', {'Maths': 90, 'Science': 75, 'Economy': 85}),
    ('Marina',   {'Science': 68, 'History': 100}),
]

schema = StructType([
    StructField('name',   StringType(),                       True),
    StructField('grades', MapType(StringType(), IntegerType()), True),
])

df_map = spark.createDataFrame(data, schema)
df_map.printSchema()
df_map.display()

In [0]:
# transform_values — scale grades to 0-20
df_map.select(
    'name',
    f.transform_values('grades', lambda k, v: v * 20 / 100).alias('grades_0_20')
).display()

# map_filter — keep only grades above 80
df_map.select(
    'name',
    f.map_filter('grades', lambda k, v: v > 80).alias('grades_above_80')
).display()

In [0]:
# Fruitshop dataset — used for transform, explode, inline, and UDFs
data = [
    {'order_id': 5642, 'order_date': datetime.strptime('2024-05-18', '%Y-%m-%d').date(),
     'items': [{'name': 'Apple', 'quantity': 1.0, 'price': 2.99},
               {'name': 'Banana', 'quantity': 1.7, 'price': 1.99}],
     'items_discount': ['Apple']},
    {'order_id': 9762, 'order_date': datetime.strptime('2024-05-02', '%Y-%m-%d').date(),
     'items': [{'name': 'Strawberry', 'quantity': 0.5, 'price': 6.99},
               {'name': 'Apple', 'quantity': 3.0, 'price': 2.99},
               {'name': 'Cherry', 'quantity': 1.0, 'price': 3.49}],
     'items_discount': ['Apple', 'Cherry']},
    {'order_id': 1234, 'order_date': datetime.strptime('2024-05-10', '%Y-%m-%d').date(),
     'items': [{'name': 'Peach', 'quantity': 2.0, 'price': 2.49}],
     'items_discount': []},
    {'order_id': 8810, 'order_date': datetime.strptime('2024-05-15', '%Y-%m-%d').date(),
     'items': [{'name': 'Banana', 'quantity': 2.5, 'price': 1.99},
               {'name': 'Peach',  'quantity': 1.0, 'price': 2.49},
               {'name': 'Apple',  'quantity': 1.0, 'price': 2.99},
               {'name': 'Cherry', 'quantity': 0.5, 'price': 3.49}],
     'items_discount': ['Banana']},
]
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, ArrayType

items_schema = ArrayType(StructType([
    StructField('name',     StringType(), True),
    StructField('quantity', DoubleType(), True),
    StructField('price',    DoubleType(), True),
]))

schema = StructType([
    StructField('order_id',       IntegerType(),           True),
    StructField('order_date',     DateType(),              True),
    StructField('items',          items_schema,            True),
    StructField('items_discount', ArrayType(StringType()), True),
])

df_fruitshop = spark.createDataFrame(data, schema)
df_fruitshop.printSchema()
df_fruitshop.display()

In [0]:
# transform — apply function to each element of an array
df_fruitshop_transformed = (
    df_fruitshop
    .withColumn('item_names', f.transform(f.col('items'), lambda x: x['name']))
)
df_fruitshop_transformed.display()

In [0]:
# explode — one row per array element
df_fruitshop_exploded = (
    df_fruitshop_transformed
    .select(
        'order_id',
        'order_date',
        f.explode(f.col('item_names')).alias('item')
    )
)
df_fruitshop_exploded.display()

In [0]:
# collect_list — inverse of explode
(
    df_fruitshop_exploded
    .groupBy('order_id', 'order_date')
    .agg(f.collect_list(f.col('item')).alias('item_names'))
).display()

In [0]:
# inline — explode array of structs into columns
(
    df_fruitshop_transformed
    .filter(f.array_contains(f.col('item_names'), 'Banana'))
    .select('order_id', f.inline(f.col('items')))
).display()

In [0]:
# Save fruitshop for exercises
df_fruitshop.write.parquet('/Volumes/workspace/default/class/fruitshop.parquet')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5551398909027507>, line 2
      1 # Save fruitshop for exercises
----> 2 df_fruitshop.write.parquet('/Volumes/workspace/default/class/fruitshop.parquet')

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/readwriter.py:755, in DataFrameWriter.parquet(self, path, mode, partitionBy, compression)
    753     self.partitionBy(partitionBy)
    754 self._set_opts(compression=compression)
--> 755 self.format("parquet").save(path)

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/readwriter.py:679, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    677     self.format(format)
    678 self._write.path = path
--> 679 _, _, ei = self._spark.client.execute_command(
    680     self._write.command(self._spark.client), self._write.observations
    681 )
    682 se

In [0]:
# UDFs vs built-ins — same task
# Built-in: transform
df_fruitshop.withColumn(
    'item_names',
    f.transform(f.col('items'), lambda x: x['name'])
).display()

# UDF equivalent
def extract_item_names(items):
    return [item['name'] for item in items]

extract_udf = f.udf(extract_item_names, ArrayType(StringType()))

df_fruitshop.withColumn(
    'item_names',
    extract_udf(f.col('items'))
).display()

In [0]:
# Regular UDF — count items per order
def count_items(items):
    return len(items)

count_items_udf = f.udf(count_items, IntegerType())

df_fruitshop.withColumn(
    'nr_items',
    count_items_udf(f.col('items'))
).select('order_id', 'nr_items').display()

In [0]:
# Complex UDF — logic built-ins can't do in one step
# avg price per unit, considering only items with quantity > 1
def avg_price_per_unit(items):
    total_price = 0.0
    total_quantity = 0.0
    for item in items:
        if item['quantity'] > 1:
            total_price += item['price'] * item['quantity']
            total_quantity += item['quantity']
    if total_quantity > 0:
        return total_price / total_quantity
    return None

avg_price_udf = f.udf(avg_price_per_unit, DoubleType())

df_fruitshop.withColumn(
    'avg_price_per_unit',
    avg_price_udf(f.col('items'))
).select('order_id', 'avg_price_per_unit').display()

In [0]:
# Load spotify data for Pandas UDFs
import urllib.request

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/afonso-pereira/spark-data/refs/heads/main/data/spotify.json',
    '/Volumes/workspace/default/class/spotify.json'
)

df_spotify = spark.read.format('json').load('/Volumes/workspace/default/class/spotify.json')
df_spotify.printSchema()
df_spotify.display()

In [0]:
import time 
# Pandas UDF — count tracks per playlist
# Compare: built-in vs regular UDF vs Pandas UDF

# Built-in (fastest — runs in JVM)
start = time.time()
df_spotify.withColumn('num_tracks', f.size(f.col('tracks'))).display()
time_builtin = time.time() - start

# Regular UDF (row by row — slow)
def count_tracks(tracks):
    return len(tracks)

count_tracks_udf = f.udf(count_tracks, IntegerType())

start = time.time()
df_spotify.withColumn('num_tracks', count_tracks_udf(f.col('tracks'))).display()
time_udf = time.time() - start

# Pandas UDF (Arrow batches — fast)
@f.pandas_udf(IntegerType())
def count_tracks_pandas(tracks: pd.Series) -> pd.Series:
    return tracks.apply(len)

start = time.time()
df_spotify.withColumn('num_tracks', count_tracks_pandas(f.col('tracks'))).display()
time_pandas_udf = time.time() - start

print(f'Built-in:   {time_builtin:.3f}s')
print(f'UDF:        {time_udf:.3f}s')
print(f'Pandas UDF: {time_pandas_udf:.3f}s')